# Failure Case Pull — GradCAM + LIME
**Owner:** Tanvi

Reads `fidelity_scores_{model}_{method}.csv`, pulls the lowest-IoU (worst
localization) images, attaches Shreya's cross-method correlation as context,
then calls `generate_overlays.py` to produce heatmap overlays for those cases.

> **Confirmed (from real files):** `method_rank_correlation_{model}.csv` is
> **per-model, not per-image** -- columns are
> `model, method_a, method_b, n, spearman_rho, p_value`. It's one Spearman
> correlation number per method pair (e.g. gradcam vs lime) across all 27
> images, computed by Shreya. So this can't be merged per-image like a
> per-image cross-check -- instead we attach the overall gradcam-vs-lime
> correlation as context alongside the failure case list (e.g. "these two
> methods agree at rho=0.55 overall, so some independent per-image failures
> are expected").
>
> **Still unverified:** `fidelity_scores_{model}_{method}.csv` hasn't been
> located yet. `IMAGE_ID_COL` / `IOU_COL` below are still best guesses --
> confirm once you find the real file (same way we found the others: search
> Drive directly, then read `df.columns.tolist()`).


## 1. Mount Google Drive (skip if your CSVs are already in the Colab runtime)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Config — EDIT THESE paths + column names to match your project

In [ ]:
import os

# Confirmed from aopc.ipynb / week2_gradcam_lime notebooks + Drive search
PROJECT_ROOT = "/content/drive/MyDrive/Projects/diabetic-retinopathy-xai"
HEATMAP_DIR = os.path.join(PROJECT_ROOT, "results/heatmaps")

# CONFIRMED: rank correlation is split per model, filename pattern below.
RANK_CORR_DIR = os.path.join(PROJECT_ROOT, "results/scores/summary")
# Real columns: model, method_a, method_b, n, spearman_rho, p_value
CORR_METHOD_A_COL = "method_a"
CORR_METHOD_B_COL = "method_b"
CORR_VALUE_COL = "spearman_rho"

# NOT yet confirmed -- still need to locate this file on Drive.
FIDELITY_DIR = os.path.join(PROJECT_ROOT, "results/scores/fidelity")  # CONFIRMED from screenshot
IMAGE_ID_COL = "image_id"   # best guess
IOU_COL = "iou"              # best guess

OUTPUT_DIR = os.path.join(PROJECT_ROOT, "results/scores/summary")
OVERLAY_SCRIPT = os.path.join(PROJECT_ROOT, "generate_overlays.py")  # confirm this exists in the repo

MODELS = ["resnet50", "efficientnetb4"]
METHODS = ["gradcam", "lime"]
TOP_N = 10   # number of worst-IoU failure cases to pull per model/method

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Fidelity dir:", FIDELITY_DIR)
print("Rank correlation dir:", RANK_CORR_DIR)
print("Output dir:", OUTPUT_DIR)


## 3. Imports

In [ ]:
import subprocess
import sys
import pandas as pd


## 4. Functions

In [ ]:
def find_fidelity_file(model, method):
    candidates = [
        os.path.join(FIDELITY_DIR, f"fidelity_scores_{model}_{method}.csv"),
        f"fidelity_scores_{model}_{method}.csv",
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    return None


def load_rank_correlation(model):
    """Loads the per-model pairwise correlation file (confirmed structure)."""
    path = os.path.join(RANK_CORR_DIR, f"method_rank_correlation_{model}.csv")
    if not os.path.exists(path):
        print(f"[WARN] {path} not found -- proceeding WITHOUT cross-method context for {model}.")
        return None
    return pd.read_csv(path)


def get_pairwise_rho(rank_corr_df, method_a, method_b):
    """Looks up the spearman_rho between two methods, regardless of column order."""
    if rank_corr_df is None:
        return None
    match = rank_corr_df[
        ((rank_corr_df[CORR_METHOD_A_COL] == method_a) & (rank_corr_df[CORR_METHOD_B_COL] == method_b)) |
        ((rank_corr_df[CORR_METHOD_A_COL] == method_b) & (rank_corr_df[CORR_METHOD_B_COL] == method_a))
    ]
    if len(match) == 0:
        return None
    return float(match[CORR_VALUE_COL].values[0])


def pick_failure_cases(fidelity_df, model, method, top_n, cross_method_rho, cross_method_partner):
    missing = [c for c in (IMAGE_ID_COL, IOU_COL) if c not in fidelity_df.columns]
    if missing:
        raise ValueError(
            f"[{model}/{method}] Missing expected column(s) {missing} in fidelity file. "
            f"Available columns: {list(fidelity_df.columns)}. "
            f"Update IMAGE_ID_COL / IOU_COL in the config cell above."
        )

    # Tiebreak with dice (also near-zero for genuine complete misses) so the
    # selection is deterministic instead of "whichever rows happened to be
    # first in the file" when many rows are tied at iou == 0.
    sort_cols = [IOU_COL] + (["dice"] if "dice" in fidelity_df.columns else [])
    sorted_df = fidelity_df.sort_values(sort_cols, ascending=True)

    n_tied_at_min = (sorted_df[IOU_COL] == sorted_df[IOU_COL].iloc[0]).sum()
    if n_tied_at_min > top_n:
        print(f"  [NOTE] {model}/{method}: {n_tied_at_min} rows tied at the minimum "
              f"{IOU_COL}={sorted_df[IOU_COL].iloc[0]}, only keeping top_n={top_n} "
              f"(tiebroken by dice). Consider raising top_n if you want all complete misses.")

    worst = sorted_df.head(top_n).copy()
    # Plain assignment (not .insert()) -- fidelity_scores csv already has its
    # own model/method columns, so .insert() collides with those.
    worst["model"] = model
    worst["method"] = method
    # Context, not a per-image merge: overall correlation between this method
    # and its counterpart (gradcam<->lime), same value on every row.
    worst[f"overall_spearman_rho_vs_{cross_method_partner}"] = cross_method_rho
    return worst


def run_overlays(model, method, image_ids):
    out_dir = os.path.join(HEATMAP_DIR, model, method, "failure_cases")
    os.makedirs(out_dir, exist_ok=True)

    if not os.path.exists(OVERLAY_SCRIPT):
        print(f"[SKIP OVERLAYS] {OVERLAY_SCRIPT} not found -- run it manually once available:")
        print(f"    python {OVERLAY_SCRIPT} --model {model} --method {method} "
              f"--image_ids {','.join(map(str, image_ids))} --out {out_dir}")
        return

    cmd = [
        sys.executable, OVERLAY_SCRIPT,
        "--model", model,
        "--method", method,
        "--image_ids", ",".join(map(str, image_ids)),
        "--out", out_dir,
    ]
    print(f"[RUN] {' '.join(cmd)}")
    try:
        subprocess.run(cmd, check=True)
    except subprocess.CalledProcessError as e:
        print(f"[ERROR] generate_overlays.py failed for {model}/{method}: {e}")


## 5. Run for all model/method combinations

In [ ]:
failure_results = {}
PARTNER = {"gradcam": "lime", "lime": "gradcam"}  # the two methods we're comparing

for model in MODELS:
    rank_corr_df = load_rank_correlation(model)

    for method in METHODS:
        fid_path = find_fidelity_file(model, method)
        if fid_path is None:
            print(f"[SKIP] {model}/{method}: 'fidelity_scores_{model}_{method}.csv' not found.")
            continue

        fidelity_df = pd.read_csv(fid_path)
        partner = PARTNER[method]
        rho = get_pairwise_rho(rank_corr_df, method, partner)
        if rho is not None:
            print(f"[CONTEXT] {model}: {method} vs {partner} overall spearman_rho = {rho:.4f}")

        try:
            failures = pick_failure_cases(fidelity_df, model, method, TOP_N, rho, partner)
        except ValueError as e:
            print(f"[ERROR] {e}")
            continue

        out_path = os.path.join(OUTPUT_DIR, f"failure_cases_{model}_{method}.csv")
        failures.to_csv(out_path, index=False)
        failure_results[(model, method)] = failures
        print(f"[OK] {model}/{method}: wrote {out_path} ({len(failures)} cases)")

        run_overlays(model, method, failures[IMAGE_ID_COL].tolist())


## 6. Peek at one result (sanity check)

In [ ]:
if failure_results:
    first_key = next(iter(failure_results))
    print(first_key)
    display(failure_results[first_key])
else:
    print("No results yet -- check the [SKIP]/[ERROR] messages above.")
